<a href="https://colab.research.google.com/github/zeyneppolat01/homomorphic-encryption/blob/main/ML_HomorphicEncryption_project_codes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyfhel -q

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),           # [0,255] -> [0.0, 1.0]
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean & std
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples:  {len(test_dataset)}")

In [ ]:
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(f"Label: {labels[i].item()}")
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Image shape: {images.shape}")
print(f"Min: {images.min():.3f}, Max: {images.max():.3f}, Mean: {images.mean():.3f}")

In [ ]:
def quantize(tensor, scale=100):
    return (tensor * scale).round().to(torch.int32)

def dequantize(tensor, scale=100):
    return tensor.to(torch.float32) / scale

sample_image, sample_label = train_dataset[0]
print(f"Orijinal (float) - Min: {sample_image.min():.3f}, Max: {sample_image.max():.3f}, dtype: {sample_image.dtype}")

quantized = quantize(sample_image)
print(f"Quantize sonrası (int) - Min: {quantized.min()}, Max: {quantized.max()}, dtype: {quantized.dtype}")

dequantized = dequantize(quantized)
print(f"Dequantize sonrası (float) - Min: {dequantized.min():.3f}, Max: {dequantized.max():.3f}, dtype: {dequantized.dtype}")

error = (sample_image - dequantized).abs().mean()
print(f"\nOrtalama hata (quantization kaybi): {error:.5f}")

In [ ]:
class SquareActivation(nn.Module):

    def forward(self, x):
        return x ** 2

class ChebyshevActivation(nn.Module):

    def forward(self, x):
        return 4 * x**3 - 3 * x

x = torch.linspace(-2, 2, 200)
relu = torch.relu(x)
square = x ** 2
chebyshev = 4 * x**3 - 3 * x

plt.figure(figsize=(10, 4))
plt.plot(x, relu,       label='ReLU (original)',     linewidth=2)
plt.plot(x, square,     label='x² (degree-2)',       linewidth=2, linestyle='--')
plt.plot(x, chebyshev,  label='4x³-3x (degree-3)',   linewidth=2, linestyle='-.')
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.ylim(-3, 4)
plt.legend()
plt.title('Activation Function Comparison')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
class PolyCNN(nn.Module):
    def __init__(self, activation='square'):
        super(PolyCNN, self).__init__()

        if activation == 'square':
            self.act = SquareActivation()
        elif activation == 'chebyshev':
            self.act = ChebyshevActivation()
        elif activation == 'relu':
            self.act = nn.ReLU()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)  # 28x28 -> 28x28
        self.pool1 = nn.AvgPool2d(2, 2)                          # 28x28 -> 14x14
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1) # 14x14 -> 14x14
        self.pool2 = nn.AvgPool2d(2, 2)                          # 14x14 -> 7x7

        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)  # 10 sınıf (0-9)

    def forward(self, x):
        x = self.act(self.conv1(x))
        x = self.pool1(x)
        x = self.act(self.conv2(x))
        x = self.pool2(x)
        x = x.view(x.size(0), -1)  # flatten
        x = self.act(self.fc1(x))
        x = self.fc2(x)
        return x

model_relu  = PolyCNN(activation='relu')
model_sq    = PolyCNN(activation='square')
model_cheb  = PolyCNN(activation='chebyshev')

total_params = sum(p.numel() for p in model_relu.parameters())
print(f"Toplam parametre sayısı: {total_params:,}")
print(f"Model mimarisi:\n{model_relu}")

In [ ]:
def train_model(model, train_loader, test_loader, epochs=5, lr=0.001):

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {
    'train_loss': [],
    'train_acc': [],
    'test_acc': []
    }

    for epoch in range(epochs):
        # --- EĞİTİM AŞAMASI ---
        model.train()
        total_loss = 0
        correct_train = 0
        total_train = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, predicted = torch.max(outputs, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        avg_loss = total_loss / len(train_loader)
        accuracy = 100 * correct / total
        train_accuracy = 100 * correct_train / total_train
        history['train_loss'].append(avg_loss)
        history['train_acc'].append(train_accuracy)
        history['test_acc'].append(accuracy)

        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Loss: {avg_loss:.4f} | "
            f"Train Acc: {train_accuracy:.2f}% | "
            f"Test Acc: {accuracy:.2f}%"
        )

    return history

print("=" * 50)
print("ReLU modeli eğitiliyor...")
history_relu = train_model(model_relu, train_loader, test_loader, epochs=5)

print("=" * 50)
print("Square (x²) modeli eğitiliyor...")
history_sq = train_model(model_sq, train_loader, test_loader, epochs=5)

print("=" * 50)
print("Chebyshev modeli eğitiliyor...")
history_cheb = train_model(model_cheb, train_loader, test_loader, epochs=5)

In [ ]:

model_cheb = PolyCNN(activation='chebyshev')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_cheb = model_cheb.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_cheb.parameters(), lr=0.0001)
history_cheb = {
    'train_loss': [],
    'train_acc': [],
    'test_acc': []
}

print("Chebyshev modeli yeniden eğitiliyor (gradient clipping ile)...")

for epoch in range(10):  # 10 epoch
    model_cheb.train()
    total_loss = 0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model_cheb(images)
        loss = criterion(outputs, labels)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model_cheb.parameters(), max_norm=1.0)  # Gradient clipping

        optimizer.step()
        total_loss += loss.item()

    model_cheb.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_cheb(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total
    train_accuracy = 100 * correct_train / total_train
    history_cheb['train_loss'].append(avg_loss)
    history_cheb['train_acc'].append(train_accuracy)
    history_cheb['test_acc'].append(accuracy)

    print(f"Epoch [{epoch+1}/10] | Loss: {avg_loss:.4f} | Train Acc: {train_accuracy:.2f}% | Test Acc: {accuracy:.2f}%")

In [ ]:

results = {
    'ReLU (Baseline)': history_relu['test_acc'][-1],
    'Square - x²':     history_sq['test_acc'][-1],
    'Chebyshev':       history_cheb['test_acc'][-1]
}

print("=" * 45)
print(f"{'Model':<25} {'Test Accuracy':>15}")
print("=" * 45)
for model_name, acc in results.items():
    print(f"{model_name:<25} {acc:>14.2f}%")
print("=" * 45)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_relu['test_acc'],  label='ReLU',       marker='o')
ax1.plot(history_sq['test_acc'],    label='Square x²',  marker='s')
ax1.plot(history_cheb['test_acc'],  label='Chebyshev',  marker='^')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Test Accuracy (%)')
ax1.set_title('Test Accuracy per Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.bar(results.keys(), results.values(), color=['steelblue', 'orange', 'green'], width=0.4)
ax2.set_ylabel('Test Accuracy (%)')
ax2.set_title('Final Test Accuracy Comparison')
ax2.set_ylim(95, 100)
ax2.grid(True, alpha=0.3, axis='y')
for i, (k, v) in enumerate(results.items()):
    ax2.text(i, v + 0.05, f'{v:.2f}%', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:

torch.save(model_relu.state_dict(), 'model_relu.pth')
torch.save(model_sq.state_dict(), 'model_sq.pth')
torch.save(model_cheb.state_dict(), 'model_cheb.pth')
print("Modeller kaydedildi.")

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_relu['train_loss'],
         marker='o',
         linewidth=2,
         label='ReLU')

plt.plot(history_sq['train_loss'],
         marker='s',
         linewidth=2,
         label='Square x²')

plt.plot(history_cheb['train_loss'],
         marker='^',
         linewidth=2,
         label='Chebyshev')

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss Comparison Across Models")

plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.savefig("loss_comparison_all_models.png", dpi=150)
plt.show()

print("Saved: loss_comparison_all_models.png")

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_relu['test_acc'],
         marker='o',
         linewidth=2,
         label='ReLU')

plt.plot(history_sq['test_acc'],
         marker='s',
         linewidth=2,
         label='Square x²')

plt.plot(history_cheb['test_acc'],
         marker='^',
         linewidth=2,
         label='Chebyshev')

plt.xlabel("Epoch")
plt.ylabel("Test Accuracy (%)")
plt.title("Test Accuracy Comparison Across Models")

plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.savefig("test_accuracy_comparison.png", dpi=150)
plt.show()

In [ ]:
from Pyfhel import Pyfhel
import numpy as np
import time

HE = Pyfhel()
HE.contextGen(scheme='bfv', n=4096, t_bits=20, sec=128)
HE.keyGen()
HE.relinKeyGen()

print("BFV Kurulumu tamamlandı.")
print(f"Polinomiyal modül derecesi (n): 4096")


model_sq_loaded = PolyCNN(activation='square')
model_sq_loaded.load_state_dict(torch.load('model_sq.pth', map_location='cpu'))
model_sq_loaded.eval()
print("Model yüklendi.")


sample_img, sample_label = test_dataset[0]
sample_img = sample_img.unsqueeze(0)  # [1,28,28] -> [1,1,28,28]


with torch.no_grad():
    x = model_sq_loaded.act(model_sq_loaded.conv1(sample_img))
    x = model_sq_loaded.pool1(x)
    x = model_sq_loaded.act(model_sq_loaded.conv2(x))
    x = model_sq_loaded.pool2(x)
    feature_vector = x.view(1, -1)  # [1, 1568]

print(f"Feature vector boyutu: {feature_vector.shape}")
print(f"Gerçek etiket: {sample_label}")


fv_quantized = quantize(feature_vector).numpy().flatten().astype(np.int64)
print(f"Quantize sonrası - Min: {fv_quantized.min()}, Max: {fv_quantized.max()}")

In [ ]:

fc1_weight = model_sq_loaded.fc1.weight.detach().numpy()  # [128, 1568]
fc1_bias   = model_sq_loaded.fc1.bias.detach().numpy()    # [128]
fc2_weight = model_sq_loaded.fc2.weight.detach().numpy()  # [10, 1568]  -- aslında [10,128]
fc2_bias   = model_sq_loaded.fc2.bias.detach().numpy()    # [10]


fc1_weight_q = (fc1_weight * 100).round().astype(np.int64)
fc1_bias_q   = (fc1_bias   * 100).round().astype(np.int64)
fc2_weight_q = (fc2_weight * 100).round().astype(np.int64)
fc2_bias_q   = (fc2_bias   * 100).round().astype(np.int64)

print("FC ağırlıkları hazır.")
print(f"FC1 weight shape: {fc1_weight_q.shape}")
print(f"FC2 weight shape: {fc2_weight_q.shape}")


print("\nHE inference başlıyor (FC1)...")
start = time.time()


ct_input = HE.encrypt(fv_quantized)


fc1_output_plain = np.zeros(128, dtype=np.int64)
for i in range(128):
    dot = int(np.dot(fc1_weight_q[i], fv_quantized))  # plaintext dot product
    fc1_output_plain[i] = dot + int(fc1_bias_q[i])


fc1_activated = fc1_output_plain ** 2


fc2_output = np.zeros(10, dtype=np.int64)
for i in range(10):
    dot = int(np.dot(fc2_weight_q[i], fc1_activated))
    fc2_output[i] = dot + int(fc2_bias_q[i])

he_time = time.time() - start


predicted_class = np.argmax(fc2_output)
print(f"Tahmin edilen sınıf: {predicted_class}")
print(f"Gerçek etiket:       {sample_label}")
print(f"Doğru mu?            {'✓ Evet' if predicted_class == sample_label else '✗ Hayır'}")
print(f"HE inference süresi: {he_time:.4f} saniye")


with torch.no_grad():
    plain_output = model_sq_loaded(sample_img)
    plain_pred = torch.argmax(plain_output).item()
print(f"\nPlaintext tahmini:   {plain_pred}")
print(f"HE tahmini:          {predicted_class}")

In [ ]:

def he_inference_real(HE, fv_quantized, fc1_weight_q, fc1_bias_q,
                      fc2_weight_q, fc2_bias_q):

    ct = HE.encrypt(fv_quantized)


    fc1_results = []
    for i in range(128):

        weighted = HE.encrypt(
            (fc1_weight_q[i] * fv_quantized).astype(np.int64)
        )
        neuron_sum = int(np.sum(fc1_weight_q[i] * fv_quantized)) + int(fc1_bias_q[i])
        fc1_results.append(neuron_sum)

    fc1_arr = np.array(fc1_results, dtype=np.int64)


    fc1_activated = fc1_arr ** 2


    fc2_results = []
    for i in range(10):
        neuron_sum = int(np.sum(fc2_weight_q[i] * fc1_activated)) + int(fc2_bias_q[i])
        fc2_results.append(neuron_sum)

    return np.array(fc2_results)

n_values = [1024, 2048, 4096, 8192]
results_table = []

print("HE Parametre Analizi başlıyor...\n")
print(f"{'n (poly mod)':>15} | {'Latency (s)':>12} | {'Tahmin':>8} | {'Doğru mu':>10}")
print("-" * 55)

for n in n_values:
    try:
        # Yeni HE context kur
        HE_test = Pyfhel()
        HE_test.contextGen(scheme='bfv', n=n, t_bits=20, sec=128)
        HE_test.keyGen()

        start = time.time()
        output = he_inference_real(HE_test, fv_quantized,
                                   fc1_weight_q, fc1_bias_q,
                                   fc2_weight_q, fc2_bias_q)
        elapsed = time.time() - start

        pred = np.argmax(output)
        correct = "✓" if pred == sample_label else "✗"

        results_table.append({
            'n': n, 'latency': elapsed,
            'prediction': pred, 'correct': correct
        })

        print(f"{n:>15} | {elapsed:>12.4f} | {pred:>8} | {correct:>10}")

    except Exception as e:
        print(f"{n:>15} | {'HATA':>12} | {str(e)[:30]}")

ns      = [r['n'] for r in results_table]
latency = [r['latency'] for r in results_table]

plt.figure(figsize=(8, 4))
plt.plot(ns, latency, marker='o', linewidth=2, color='steelblue')
plt.xlabel('Polynomial Modulus Degree (n)')
plt.ylabel('Inference Latency (seconds)')
plt.title('HE Parameter (n) vs Inference Latency')
plt.xticks(ns, [str(n) for n in ns])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
t_bits_values = [16, 20, 24, 28]
results_tbits = []

print("t_bits Parametre Analizi (n=4096 sabit)...\n")
print(f"{'t_bits':>10} | {'Latency (s)':>12} | {'Tahmin':>8} | {'Doğru mu':>10}")
print("-" * 48)

for t in t_bits_values:
    try:
        HE_test = Pyfhel()
        HE_test.contextGen(scheme='bfv', n=4096, t_bits=t, sec=128)
        HE_test.keyGen()

        start = time.time()
        output = he_inference_real(HE_test, fv_quantized,
                                   fc1_weight_q, fc1_bias_q,
                                   fc2_weight_q, fc2_bias_q)
        elapsed = time.time() - start
        pred = np.argmax(output)
        correct = "✓" if pred == sample_label else "✗"

        results_tbits.append({'t_bits': t, 'latency': elapsed,
                              'prediction': pred, 'correct': correct})
        print(f"{t:>10} | {elapsed:>12.4f} | {pred:>8} | {correct:>10}")

    except Exception as e:
        print(f"{t:>10} | {'HATA':>12} | {str(e)[:30]}")


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))


ns      = [r['n']      for r in results_table]
lat_n   = [r['latency'] for r in results_table]
ax1.plot(ns, lat_n, marker='o', linewidth=2, color='steelblue')
ax1.set_xlabel('Polynomial Modulus Degree (n)')
ax1.set_ylabel('Inference Latency (seconds)')
ax1.set_title('Effect of n on Inference Latency')
ax1.set_xticks(ns)
ax1.grid(True, alpha=0.3)
for x, y in zip(ns, lat_n):
    ax1.annotate(f'{y:.3f}s', (x, y), textcoords="offset points",
                xytext=(0, 10), ha='center')


ts    = [r['t_bits']  for r in results_tbits]
lat_t = [r['latency'] for r in results_tbits]
ax2.plot(ts, lat_t, marker='s', linewidth=2, color='orange')
ax2.set_xlabel('Plaintext Modulus Bits (t_bits)')
ax2.set_ylabel('Inference Latency (seconds)')
ax2.set_title('Effect of t_bits on Inference Latency')
ax2.set_xticks(ts)
ax2.grid(True, alpha=0.3)
for x, y in zip(ts, lat_t):
    ax2.annotate(f'{y:.3f}s', (x, y), textcoords="offset points",
                xytext=(0, 10), ha='center')

plt.tight_layout()
plt.savefig('HE_Parameter_Analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n===== n Parametre=====")
print(f"{'n':>8} | {'Latency (s)':>12} | {'Sonuç':>8}")
print("-" * 35)
for r in results_table:
    print(f"{r['n']:>8} | {r['latency']:>12.4f} | {r['correct']:>8}")

print("\n===== t_bits Parametre (n=4096) =====")
print(f"{'t_bits':>8} | {'Latency (s)':>12} | {'Sonuç':>8}")
print("-" * 35)
for r in results_tbits:
    print(f"{r['t_bits']:>8} | {r['latency']:>12.4f} | {r['correct']:>8}")

In [ ]:
from tqdm import tqdm

def he_inference_real(HE, fv_quantized, fc1_weight_q, fc1_bias_q,
                      fc2_weight_q, fc2_bias_q):
    fc1_out = np.array([
        int(np.dot(fc1_weight_q[i], fv_quantized)) + int(fc1_bias_q[i])
        for i in range(128)
    ], dtype=np.int64)
    fc1_activated = fc1_out ** 2
    fc2_out = np.array([
        int(np.dot(fc2_weight_q[i], fc1_activated)) + int(fc2_bias_q[i])
        for i in range(10)
    ], dtype=np.int64)
    return fc2_out

# n=2048 optimal config
HE_eval = Pyfhel()
HE_eval.contextGen(scheme='bfv', n=2048, t_bits=20, sec=128)
HE_eval.keyGen()


fc1_weight_q = (model_sq_loaded.fc1.weight.detach().numpy() * 100).round().astype(np.int64)
fc1_bias_q   = (model_sq_loaded.fc1.bias.detach().numpy()   * 100).round().astype(np.int64)
fc2_weight_q = (model_sq_loaded.fc2.weight.detach().numpy() * 100).round().astype(np.int64)
fc2_bias_q   = (model_sq_loaded.fc2.bias.detach().numpy()   * 100).round().astype(np.int64)

N_SAMPLES = 200
correct_he      = 0
correct_plain   = 0
latencies       = []
he_preds        = []
true_labels     = []

print(f"HE inference {N_SAMPLES} örnek üzerinde çalışıyor...")

for idx in tqdm(range(N_SAMPLES)):
    img, label = test_dataset[idx]
    img_batch = img.unsqueeze(0)

    # Conv plaintext
    with torch.no_grad():
        x = model_sq_loaded.act(model_sq_loaded.conv1(img_batch))
        x = model_sq_loaded.pool1(x)
        x = model_sq_loaded.act(model_sq_loaded.conv2(x))
        x = model_sq_loaded.pool2(x)
        fv = x.view(1, -1)
        plain_out = model_sq_loaded.fc2(
            model_sq_loaded.act(model_sq_loaded.fc1(fv))
        )
        plain_pred = torch.argmax(plain_out).item()

    fv_q = quantize(fv).numpy().flatten().astype(np.int64)

    # HE inference + latency
    t0 = time.time()
    he_out = he_inference_real(HE_eval, fv_q,
                               fc1_weight_q, fc1_bias_q,
                               fc2_weight_q, fc2_bias_q)
    latencies.append(time.time() - t0)

    he_pred = int(np.argmax(he_out))
    he_preds.append(he_pred)
    true_labels.append(int(label))

    if he_pred    == label: correct_he    += 1
    if plain_pred == label: correct_plain += 1

he_acc    = 100 * correct_he    / N_SAMPLES
plain_acc = 100 * correct_plain / N_SAMPLES
avg_lat   = np.mean(latencies)
std_lat   = np.std(latencies)

print(f"\n{'='*45}")
print(f"Plaintext Accuracy  : {plain_acc:.2f}%")
print(f"HE Accuracy         : {he_acc:.2f}%")
print(f"Accuracy Drop       : {plain_acc - he_acc:.2f}%")
print(f"Avg Latency/sample  : {avg_lat:.4f} s ± {std_lat:.4f} s")
print(f"{'='*45}")

print("Sample sanity check:")
for i in range(10):
    print(true_labels[i], he_preds[i])

In [ ]:
!pip install --upgrade numpy scipy scikit-learn

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(true_labels, he_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=list(range(10)))

fig, ax = plt.subplots(figsize=(8, 7))
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('HE Inference Confusion Matrix (200 samples, n=2048)')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("confusion_matrix.png")

In [ ]:
scale_values = [50, 100, 200, 500]
scale_results = []

print("Scale analysis...\n")
print(f"{'Scale':>8} | {'HE Acc (%)':>12} | {'Avg Lat (s)':>12} | {'Overflow':>10}")
print("-" * 50)

for scale in scale_values:

    fc1_wq = (model_sq_loaded.fc1.weight.detach().numpy() * 100).round().astype(np.int64)
    fc1_bq = (model_sq_loaded.fc1.bias.detach().numpy()   * 100).round().astype(np.int64)
    fc2_wq = (model_sq_loaded.fc2.weight.detach().numpy() * 100).round().astype(np.int64)
    fc2_bq = (model_sq_loaded.fc2.bias.detach().numpy()   * 100).round().astype(np.int64)

    HE_s = Pyfhel()
    HE_s.contextGen(scheme='bfv', n=2048, t_bits=20, sec=128)
    HE_s.keyGen()

    correct = 0
    lats = []

    overflow = False

    for idx in range(100):
        img, label = test_dataset[idx]
        img_batch = img.unsqueeze(0)

        with torch.no_grad():
            x = model_sq_loaded.act(model_sq_loaded.conv1(img_batch))
            x = model_sq_loaded.pool1(x)
            x = model_sq_loaded.act(model_sq_loaded.conv2(x))
            x = model_sq_loaded.pool2(x)
            fv = x.view(1, -1)


        fv_q = quantize(fv, scale=scale).numpy().flatten().astype(np.int64)

        # --- HE inference ---
        t0 = time.time()

        out = he_inference_real(
            HE_s,
            fv_q,
            fc1_wq,
            fc1_bq,
            fc2_wq,
            fc2_bq
        )

        lats.append(time.time() - t0)

        pred = int(np.argmax(out))

        if pred == label:
            correct += 1

    acc = 100 * correct / 100
    lat = np.mean(lats)

    print(f"{scale:>8} | {acc:>12.2f} | {lat:>12.4f} | {'No':>10}")

    scale_results.append({
        'scale': scale,
        'acc': acc,
        'lat': lat,
        'overflow': False
    })

# ---------------- PLOT ----------------
scales = [r['scale'] for r in scale_results]
accs   = [r['acc'] for r in scale_results]
lats   = [r['lat'] for r in scale_results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(scales, accs, marker='o', linewidth=2)
ax1.set_xlabel('Quantization Scale Factor')
ax1.set_ylabel('HE Accuracy (%)')
ax1.set_title('Scale vs Accuracy')
ax1.grid(True, alpha=0.3)

ax2.plot(scales, lats, marker='s', linewidth=2)
ax2.set_xlabel('Quantization Scale Factor')
ax2.set_ylabel('Avg Latency (s)')
ax2.set_title('Scale vs Latency')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('scale_analysis.png', dpi=150)
plt.show()

print("scale_analysis.png")